# Capa Silver: Enriquecimiento y Cruces (JOINs) Masivos

En la arquitectura Medallón, la capa **Silver** toma los datos limpios y deduplicados de la capa Bronze, y aplica lógica de negocio, cruces de tablas (JOINs) y limpieza de tipos de datos.

En este ejercicio, tomaremos la tabla maestra de Pedidos (`orders`), la cruzaremos con el detalle masivo de compras (`lineitem`) y con el catálogo de clientes (`customer`) para crear una tabla desnormalizada llamada **`enriched_sales`**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, year, month

spark = (SparkSession.builder
    .appName("TPCH_Bronze_to_Silver")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "2")
    .getOrCreate())

print("Spark Session conectada!")

## 1. Lectura de las Tablas
Leemos `orders` (desde nuestra capa Bronze) y `lineitem` y `customer` (desde los archivos Parquet generados).

In [ ]:
base_path = "/opt/spark/data"

# 1. Orders (desde Bronze)
df_orders = spark.read.parquet(f"{base_path}/bronze/orders")

# 2. Lineitem (Tabla gigante de detalles)
df_lineitem = spark.read.parquet(f"{base_path}/source/tpch/lineitem.parquet")

# 3. Customer (Clientes)
df_customer = spark.read.parquet(f"{base_path}/source/tpch/customer.parquet")

print(f"Órdenes: {df_orders.count()} | Detalle: {df_lineitem.count()} | Clientes: {df_customer.count()}")

## 2. El Cruce Masivo (JOIN)
Vamos a unir las tres tablas para tener todo el contexto de una venta en un solo lugar. Spark distribuirá este esfuerzo usando un algoritmo llamado *SortMergeJoin* o *BroadcastHashJoin* dependiendo del tamaño.

In [ ]:
# Hacemos el JOIN de Pedidos con su Detalle
df_ventas = df_lineitem.join(df_orders, df_lineitem.l_orderkey == df_orders.o_orderkey, "inner")

# Y luego le pegamos el Cliente
df_ventas_cliente = df_ventas.join(df_customer, df_orders.o_custkey == df_customer.c_custkey, "inner")

print("JOIN lógico construido (Aún no procesado por ser Lazy Evaluation)")

## 3. Lógica de Negocio (Enriquecimiento)
Un requerimiento clásico: TPC-H nos da el precio unitario, el descuento y el impuesto. El negocio nos pide calcular el **Ingreso Neto Real** por cada ítem vendido.

Fórmula: `l_extendedprice * (1 - l_discount) * (1 + l_tax)`

In [ ]:
df_silver = df_ventas_cliente.select(
    col("o_orderkey").alias("order_id"),
    col("o_orderdate").alias("order_date"),
    col("c_name").alias("customer_name"),
    col("c_mktsegment").alias("market_segment"),
    col("l_partkey").alias("product_id"),
    col("l_quantity").alias("quantity"),
    col("l_extendedprice").alias("base_price"),
    col("l_discount").alias("discount_rate"),
    # Cálculo del Ingreso Neto Real
    expr("l_extendedprice * (1 - l_discount) * (1 + l_tax)").alias("net_revenue")
)

# Mostramos las primeras filas (Esto dispara la ejecución del JOIN en los workers)
df_silver.show(5)

## 4. Particionado y Escritura en Silver
Para que la capa Gold sea súper rápida al generar reportes mensuales, vamos a extraer el Año y el Mes de la fecha del pedido, y guardaremos los datos particionados físicamente en carpetas por año y mes.

In [ ]:
silver_path = f"{base_path}/silver/enriched_sales"

# Extraemos Año y Mes para usarlos como particiones
df_silver_partitioned = df_silver \
    .withColumn("order_year", year(col("order_date"))) \
    .withColumn("order_month", month(col("order_date")))

# Guardamos en formato Parquet
df_silver_partitioned.write \
    .mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .parquet(silver_path)

print(f"Capa Silver generada exitosamente en: {silver_path}")

In [ ]:
# Mantendremos la sesión viva para que puedas inspeccionar el puerto 4040 si lo deseas
# spark.stop()